# High-quality videos at FULL resolution, no trained models of ours

Everything runs on the original 1920x1080 frames (nothing is shrunk):
- skier: COCO YOLO11 + ByteTrack, skeleton: COCO YOLO11-pose, ski: YOLOE "water ski" at the feet
- buoys: multi-frame. YOLOE "buoy" + colour centre-surround with a CLIP check find candidates; each buoy is then
  searched only near its last position in the next frames, and counts only if it persists (>= 4 frames)
- pass logic v4 (skier movement + buoy proximity)

**Runtime → Change runtime type → A100 GPU** (T4 works too, about 2x slower) **→ Run all.**
Estimated time on an A100: about 35–45 min in total (step 2: about 5 min, step 3: about 30–40 min).
At the end `hq_full.zip` downloads: upload it to Drive and send the link.

In [ ]:
import os
if not os.path.exists('/content/CV_test'):
    !git clone --depth 1 -b main https://github.com/lytvynenkomaksim/CV_test.git /content/CV_test
%cd /content/CV_test
!git log --oneline -1
!pip install -q ultralytics tqdm lap ftfy regex git+https://github.com/openai/CLIP.git
!mkdir -p models && cd models && for w in yolo11s.pt yolo11s-pose.pt; do [ -f $w ] || wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/$w; done
!cd models && [ -f yoloe-26l-seg.pt ] || wget -q https://github.com/ultralytics/assets/releases/download/v8.4.0/yoloe-26l-seg.pt
!yolo settings weights_dir=/content/CV_test/models > /dev/null
import clip; clip.load('ViT-B/32', download_root='models/clip')
!nvidia-smi --query-gpu=name --format=csv

## 2. Quality check: the multi-frame detector on the 40 hand-checked frames (28 buoys)

In [ ]:
!python tools/eval_temporal.py --gold data/buoy_gold_hq --out results/zeroshot_hq/preds
!python tools/zeroshot_score.py --gold data/buoy_gold_hq --preds results/zeroshot_hq/preds --out results/zeroshot_hq --no-heldout
from IPython.display import Markdown; display(Markdown(open('results/zeroshot_hq/summary.md').read()))

## 3. All 7 high-quality videos: buoys, ski, skeleton, pass / miss, result videos

In [ ]:
!python tools/run_hq.py data/videos/drive_extra/ItalyPro_OpenWomenRd2__Julie_Mishler_6at32_Rd2.mp4 data/videos/drive_extra/ItalyPro_HeadToHead__Allie_Nicholson_6at38.mp4 data/videos/drive_extra/ItalyPro_OpenWomenRd2__Saba_York_Davis_6at13_Rd2.mp4 data/videos/drive_extra/ItalyPro_HeadToHead__Beatrice_Ianni_4at38.mp4 data/videos/drive_extra/ItalyPro_MenFinal__Nate_Smith_4at41.mp4 data/videos/drive_extra/ItalyPro_MenFinal__Saha_Descuns_4at41_semi_final.mp4 data/videos/youtube/youtube_regina_jaquess.mp4 --out results/hq_full
display(Markdown(open('results/hq_full/summary.md').read()))

In [ ]:
!zip -qr hq_full.zip results/hq_full results/zeroshot_hq
!ls -la hq_full.zip
from google.colab import files; files.download('hq_full.zip')